# Project 3

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import heapq
import pandas as pd
import seaborn as sns
from scipy.stats import norm

In [ ]:
class Ward:
    def __init__(self, name, arrival_func, u, std, beds=None):
        self.name = name
        self.arrival_func = arrival_func 
        self.u = u
        self.std = std
        self.beds = beds

    def arrival_rate(self, t):
        return self.arrival_func(t)

    def length_of_stay(self):
        return np.random.lognormal(self.u, self.std)

std = np.sqrt(np.log(2))

arrival_a = lambda t: -(1/3650) * t**2 + (1/10) * t
arrival_b = lambda t: (1/5) * arrival_a(t)
arrival_c = lambda t: 6.0 
mu_a = np.log(4 * np.sqrt(2))
mu_b = np.log(6 * np.sqrt(2))
mu_c = np.log(5 * np.sqrt(2))


## Primary task

In [ ]:
def get_all_bed_distributions(total_beds):
    return [
        (a, b, total_beds - a - b)
        for a in range(1, total_beds - 1)
        for b in range(1, total_beds - a)
    ]


def simulate_system(*wards, n_days):
    occupancy = [[] for _ in wards]
    stats = {
        "relocated": [0] * len(wards),
        "full": [0] * len(wards),
        "arrivals": [0] * len(wards),
        "history": [[] for _ in wards]
    }

    for day in range(n_days):
        for occ in occupancy:
            occ[:] = [x for x in occ if x > day]

        arrivals = [
            np.random.poisson(w.arrival_rate(day))
            for w in wards
        ]

        for i, (ward, amount) in enumerate(zip(wards, arrivals)):
            stats["arrivals"][i] += amount

            for _ in range(amount):
                if len(occupancy[i]) >= ward.beds:
                    stats["full"][i] += 1

                if len(occupancy[i]) < ward.beds:
                    occupancy[i].append(day + ward.length_of_stay())

                elif i == 1 and len(occupancy[0]) < wards[0].beds:
                    occupancy[0].append(day + ward.length_of_stay())

                else:
                    stats["relocated"][i] += 1

        for i, occ in enumerate(occupancy):
            stats["history"][i].append(len(occ))

    return {
        "relocations": {
            "A": stats["relocated"][0],
            "B": stats["relocated"][1],
            "C": stats["relocated"][2],
            "Total": sum(stats["relocated"])
        },
        "prob_full_on_arrival": {
            k: stats["full"][i] / stats["arrivals"][i] if stats["arrivals"][i] else 0
            for i, k in enumerate("ABC")
        },
        "mean_utilization": {
            k: np.mean(stats["history"][i]) / wards[i].beds
            for i, k in enumerate("ABC")
        },
        "history": stats["history"]
    }


def find_optimal_distribution(distributions, *wards, replications=3):
    best = None
    lowest = float("inf")

    print(f"Testing {len(distributions)} bed distributions...")

    for beds in distributions:
        for ward, beds_amount in zip(wards, beds):
            ward.beds = beds_amount

        total = 0

        for _ in range(replications):
            np.random.seed(42)
            total += simulate_system(*wards, n_days=365)["relocations"]["Total"]

        avg = total / replications

        if avg < lowest:
            lowest = avg
            best = beds

    return best, lowest

In [ ]:
np.random.seed(42)

n = 365
total_beds = 75

wards = [
    Ward("Ward A", arrival_func=arrival_a, u=mu_a, std=std, beds=0),
    Ward("Ward B", arrival_func=arrival_b, u=mu_b, std=std, beds=0),
    Ward("Ward C", arrival_func=arrival_c, u=mu_c, std=std, beds=0)
]

dist = get_all_bed_distributions(total_beds)

optimal_beds, min_relocated = find_optimal_distribution(
    dist, *wards, replications=1
)

print("\n--- OPTIMAL CONFIGURATION FOUND ---")
print("\n".join(
    f"Ward {chr(65+i)} Beds: {beds}"
    for i, beds in enumerate(optimal_beds)
))
print(f"Minimum Average Relocated Patients: {min_relocated:.2f}")

for ward, beds in zip(wards, optimal_beds):
    ward.beds = beds

print("\nFinal Results for Optimal Distribution:")
final = simulate_system(*wards, n_days=n)
print(final)

In [ ]:
def simulate_system_steady(*wards, n_days, warmup_days=200, cooldown_days=200):
    total_days = warmup_days + n_days + cooldown_days

    occupancy = [[] for _ in wards]
    stats = {
        "relocated": [0] * len(wards),
        "full": [0] * len(wards),
        "arrivals": [0] * len(wards),
        "history": [[] for _ in wards]
    }

    for day in range(total_days):

        # Fjern udskrevne patienter
        for occ in occupancy:
            occ[:] = [x for x in occ if x > day]

        # Generér ankomster
        # arrivals = [
        #     np.random.poisson(max(0, w.arrival_rate(day)))
        #     for w in wards
        # ]

        arrivals = [
            np.random.poisson(
                max(0, w.arrival_rate((day - warmup_days) % 365))
            )
            for w in wards
        ]


        # Er vi i måleperioden?
        in_main = warmup_days <= day < warmup_days + n_days

        for i, (ward, amount) in enumerate(zip(wards, arrivals)):
            if in_main:
                stats["arrivals"][i] += amount

            for _ in range(amount):

                ward_full = len(occupancy[i]) >= ward.beds

                if ward_full and in_main:
                    stats["full"][i] += 1

                # Normal indlæggelse
                if len(occupancy[i]) < ward.beds:
                    occupancy[i].append(day + ward.length_of_stay())

                # B → A relocation
                elif i == 1 and len(occupancy[0]) < wards[0].beds:
                    occupancy[0].append(day + ward.length_of_stay())

                # Relocation
                else:
                    if in_main:
                        stats["relocated"][i] += 1

        # Gem kun belægning i måleperioden
        if in_main:
            for i, occ in enumerate(occupancy):
                stats["history"][i].append(len(occ))

    return {
        "relocations": {
            "A": stats["relocated"][0],
            "B": stats["relocated"][1],
            "C": stats["relocated"][2],
            "Total": sum(stats["relocated"])
        },
        "prob_full_on_arrival": {
            k: stats["full"][i] / stats["arrivals"][i] if stats["arrivals"][i] else 0
            for i, k in enumerate("ABC")
        },
        "mean_utilization": {
            k: np.mean(stats["history"][i]) / wards[i].beds
            for i, k in enumerate("ABC")
        },
        "history": stats["history"]
    }

def find_optimal_distribution_steady(distributions, *wards,
                                     replications=1,
                                     warmup_days=200,
                                     cooldown_days=200,
                                     n_days=365):

    best = None
    lowest = float("inf")

    print(f"Testing {len(distributions)} bed distributions (steady state)...")

    for beds in distributions:
        # Sæt senge
        for ward, beds_amount in zip(wards, beds):
            ward.beds = beds_amount

        total = 0

        for _ in range(replications):
            np.random.seed(42)
            result = simulate_system_steady(
                *wards,
                n_days=n_days,
                warmup_days=warmup_days,
                cooldown_days=cooldown_days
            )
            total += result["relocations"]["Total"]

        avg = total / replications

        if avg < lowest:
            lowest = avg
            best = beds

    return best, lowest


In [ ]:
np.random.seed(42)

n = 365
total_beds = 75

wards = [
    Ward("Ward A", arrival_func=arrival_a, u=mu_a, std=std, beds=0),
    Ward("Ward B", arrival_func=arrival_b, u=mu_b, std=std, beds=0),
    Ward("Ward C", arrival_func=arrival_c, u=mu_c, std=std, beds=0)
]

dist = get_all_bed_distributions(total_beds)

# Finite-horizon optimization (starting at nothing)
optimal_beds_finite, min_relocated_finite = find_optimal_distribution(
    dist, *wards, replications=1
)

print("\n--- OPTIMAL CONFIGURATION FOUND (Finite Horizon) ---")
for i, beds in enumerate(optimal_beds_finite):
    print(f"Ward {chr(65+i)} Beds: {beds}")
print(f"Minimum Average Relocated Patients: {min_relocated_finite:.2f}")

for ward, beds in zip(wards, optimal_beds_finite):
    ward.beds = beds

final_finite = simulate_system(*wards, n_days=n)

print("\n--- FINAL RESULTS (Finite Horizon) ---")
print(final_finite)

# Steady-state optimization (warm-uo/cooldown periods)
for ward in wards:
    ward.beds = 0

optimal_beds_steady, min_relocated_steady = find_optimal_distribution_steady(
    dist, *wards, replications=1,
    warmup_days=200, cooldown_days=200, n_days=365
)

print("\n--- OPTIMAL CONFIGURATION FOUND (Steady State) ---")
for i, beds in enumerate(optimal_beds_steady):
    print(f"Ward {chr(65+i)} Beds: {beds}")
print(f"Minimum Average Relocated Patients (steady): {min_relocated_steady:.2f}")

for ward, beds in zip(wards, optimal_beds_steady):
    ward.beds = beds

steady_results = simulate_system_steady(
    *wards, n_days=n, warmup_days=200, cooldown_days=200
)

print("\n--- FINAL RESULTS (Steady State) ---")
print(steady_results)


In [ ]:
np.random.seed(42)
finite_results_all = []
steady_results_all = []

for beds in dist:
    # Finite horizon
    for ward, beds_amount in zip(wards, beds):
        ward.beds = beds_amount
    res_finite = simulate_system(*wards, n_days=365)
    finite_results_all.append(res_finite["relocations"]["Total"])

# Reset beds
for ward in wards:
    ward.beds = 0

for beds in dist:
    # Steady state
    for ward, beds_amount in zip(wards, beds):
        ward.beds = beds_amount
    res_steady = simulate_system_steady(*wards, n_days=365, warmup_days=200, cooldown_days=200)
    steady_results_all.append(res_steady["relocations"]["Total"])


In [ ]:
import numpy as np
from scipy.stats import chisquare

# Convert to numpy
finite = np.array(finite_results_all)
steady = np.array(steady_results_all)

# Define bins
bins = np.linspace(min(finite.min(), steady.min()),
                   max(finite.max(), steady.max()),
                   20)

finite_hist, _ = np.histogram(finite, bins=bins)
steady_hist, _ = np.histogram(steady, bins=bins)

chi2, p = chisquare(f_obs=finite_hist, f_exp=steady_hist)

print("Chi-square statistic:", chi2)
print("p-value:", p)

from scipy.stats import ks_2samp

ks_stat, ks_p = ks_2samp(finite, steady)

print("KS statistic:", ks_stat)
print("p-value:", ks_p)

In [ ]:
from scipy.stats import ks_2samp

ks_stat, ks_p = ks_2samp(finite, steady)

print("KS statistic:", ks_stat)
print("p-value:", ks_p)

# Sammenligning
\subsection{Steady--State Model With Warm--up and Cool--down}

To obtain performance measures representative of long-run system behaviour, a steady--state simulation model is implemented. In contrast to the finite-horizon setup, the system does not start empty at the beginning of the measurement period. Instead, the simulation includes a warm--up period of $200$ days, followed by a $365$-day measurement window, and finally a cool--down period of $200$ days. Only observations collected during the central measurement window are used for performance estimation.

The total simulated horizon is therefore


\[
T = T_{\text{warm}} + T_{\text{main}} + T_{\text{cool}} 
= 200 + 365 + 200 = 765 \text{ days}.
\]



During the warm--up and cool--down periods, arrivals and discharges evolve normally, but no statistics are recorded. This ensures that the occupancy distribution at the start of the measurement window is representative of a system in equilibrium, thereby eliminating boundary effects caused by artificially empty initial conditions.

Arrivals are generated using the same time-dependent arrival rates as in the finite-horizon model, but with periodicity enforced through a modulo operation:


\[
\lambda_w(t) = \lambda_w\!\left( (t - T_{\text{warm}}) \bmod 365 \right),
\]


ensuring that seasonal patterns repeat consistently throughout the extended simulation horizon. Daily arrivals are sampled from Poisson distributions, and lengths of stay follow the same lognormal distributions as before.

Capacity constraints and relocation rules are identical to the finite-horizon model. Patients assigned to Ward B may be redirected to Ward A if capacity is available; otherwise, the patient is counted as relocated. Events such as ``full on arrival'', relocations, and occupancy histories are recorded only during the measurement window.

To determine the optimal bed allocation under steady-state conditions, all feasible distributions $(a,b,c)$ with $a+b+c=75$ are evaluated. For each candidate distribution, the steady-state simulation is executed, and the total number of relocations observed during the measurement window is recorded. The optimal configuration is defined as the allocation that minimizes the expected number of relocations across replications.

\begin{table}[H]
\centering

\begin{minipage}{0.48\textwidth}
\centering
\begin{tabular}{lcccc}
\toprule
\textbf{Ward} & \textbf{Beds} & \textbf{Reloc.} & \textbf{Prob(full)} & \textbf{Util.} \\
\midrule
A & 41 & 848  & 0.370 & 0.865 \\
B & 2  & 285  & 0.853 & 0.911 \\
C & 32 & 1067 & 0.478 & 0.982 \\
\midrule
\textbf{Total} & 75 & \textbf{2200} & -- & -- \\
\bottomrule
\end{tabular}
\caption*{\textbf{Finite Horizon}}
\end{minipage}
\hfill
\begin{minipage}{0.48\textwidth}
\centering
\begin{tabular}{lcccc}
\toprule
\textbf{Ward} & \textbf{Beds} & \textbf{Reloc.} & \textbf{Prob(full)} & \textbf{Util.} \\
\midrule
A & 39 & 862  & 0.385 & 0.884 \\
B & 1  & 315  & 0.927 & 0.932 \\
C & 35 & 941  & 0.442 & 0.986 \\
\midrule
\textbf{Total} & 75 & \textbf{2118} & -- & -- \\
\bottomrule
\end{tabular}
\caption*{\textbf{Steady State (Warm--up + Cool--down)}}
\end{minipage}

\caption{Side-by-side comparison of optimal bed allocations and performance metrics under the finite-horizon and steady-state models.}
\label{tab:side_by_side}
\end{table}

Table~\ref{tab:side_by_side} presents a direct comparison of the optimal bed allocations and performance metrics obtained under the finite-horizon and steady-state models. Although both models allocate the majority of beds to Wards~A and~C, the resulting optimal configurations differ in meaningful ways.

In the finite-horizon model, the system begins empty on day~0, which creates boundary effects that favour Ward~B receiving slightly more capacity. This leads to an optimal allocation of $(41, 2, 32)$ beds for Wards~A, B, and C, respectively. The total number of relocated patients during the simulated year is $2200$, with Ward~C contributing the largest share due to its high and constant arrival rate.

In contrast, the steady-state model incorporates a warm--up and cool--down period, ensuring that the occupancy distribution at the start of the measurement window reflects long-run equilibrium behaviour. This shifts the optimal allocation toward $(39, 1, 35)$ beds, reducing Ward~B to a single bed while increasing Ward~C’s capacity. The total number of relocations decreases to $2118$, indicating that the steady-state configuration better accommodates long-run patient flows.

Across both models, Ward~C consistently exhibits the highest utilization levels (above $0.98$), reflecting its constant arrival intensity. Ward~B shows the highest probability of being full on arrival in both scenarios, particularly in the steady-state model, where the probability exceeds $0.92$. Ward~A maintains high utilization in both models, though slightly higher in steady state due to the more balanced long-run occupancy distribution.

Overall, the comparison highlights that the finite-horizon model is more sensitive to initial conditions, whereas the steady-state model provides a more realistic representation of long-run operational performance. The shift in optimal bed allocation from Ward~B to Ward~C in the steady-state model reflects the system’s natural equilibrium behaviour once transient effects are removed.

\subsection{Statistical Comparison Across All Bed Distributions}

To assess whether the finite-horizon and steady-state models produce systematically different outcomes across the entire parameter space, we performed a statistical comparison based on all $2701$ feasible bed distributions. For each allocation $(a,b,c)$ with $a+b+c=75$, both models were simulated and the total number of relocated patients was recorded. This yields two empirical distributions of relocation counts, one for each modelling approach.

Two statistical tests were applied. First, a chi-square goodness-of-fit test was conducted by binning the relocation counts into $20$ intervals and comparing the resulting histograms. Second, a two-sample Kolmogorov--Smirnov (KS) test was used to compare the empirical cumulative distribution functions directly, without requiring binning.

The chi-square test produced a test statistic of


\[
\chi^2 = 22.64,
\qquad
p = 0.205,
\]


indicating no statistically significant difference between the binned relocation distributions at the $5\%$ significance level. The KS test similarly found no evidence of distributional differences, yielding


\[
D = 0.027,
\qquad
p = 0.277.
\]



Taken together, these results suggest that although the two models yield different optimal bed allocations and differ in their detailed performance metrics, their overall relocation behaviour across the full space of bed distributions is statistically similar. In other words, the finite-horizon and steady-state models exhibit comparable global relocation patterns when evaluated over all feasible capacity configurations, even though their optimal solutions differ.



In [ ]:
np.random.seed(42)
final_steady = simulate_system_steady(*wards, n_days=n, warmup_days=200, cooldown_days=200)
history_steady = final_steady["history"]

df_history_steady = pd.DataFrame({
    "A": history_steady[0],
    "B": history_steady[1],
    "C": history_steady[2]
})

plt.figure(figsize=(12,6))
plt.plot(df_history_steady["A"], label="Ward A")
plt.plot(df_history_steady["B"], label="Ward B")
plt.plot(df_history_steady["C"], label="Ward C")
plt.xlabel("Day")
plt.ylabel("Occupied beds")
plt.title("Daily Occupancy for Optimal Bed Distribution")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
np.random.seed(42)
final = simulate_system(*wards, n_days=n)
history = final["history"]

df_history = pd.DataFrame({
    "A": history[0],
    "B": history[1],
    "C": history[2]
})

plt.figure(figsize=(12,6))
plt.plot(df_history["A"], label="Ward A")
plt.plot(df_history["B"], label="Ward B")
plt.plot(df_history["C"], label="Ward C")
plt.xlabel("Day")
plt.ylabel("Occupied beds")
plt.title("Daily Occupancy for Optimal Bed Distribution")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
pfull = final["prob_full_on_arrival"]

plt.figure(figsize=(6,5))
plt.bar(["Ward A", "Ward B", "Ward C"],
        [pfull["A"], pfull["B"], pfull["C"]],
        color=["#4C72B0", "#55A868", "#C44E52"])
plt.ylabel("Probability")
plt.title("Probability of Full Ward on Arrival")
plt.show()


In [ ]:
rel = final["relocations"]

plt.figure(figsize=(6,5))
plt.bar(["Ward A", "Ward B", "Ward C"],
        [rel["A"], rel["B"], rel["C"]],
        color=["#4C72B0", "#55A868", "#C44E52"])
plt.ylabel("Relocated Patients")
plt.title("Relocations per Ward")
plt.show()


## Primary performance measures

In [ ]:
def run_replications(wards, n_days, R):
    results = []

    for r in range(R):
        # vigtigt: ny seed eller ingen seed her, så replications varierer
        res = simulate_system(*wards, n_days=n_days)
        results.append(res)

    return results


In [ ]:
random_seed = 42
np.random.seed(random_seed)
for ward, beds in zip(wards, optimal_beds):
    ward.beds = beds

R = 500
results = run_replications(wards, n_days=n, R=R)

In [ ]:
pfull = {k: [] for k in "ABC"}
rel = {k: [] for k in ["A", "B", "C", "Total"]}
util = {k: [] for k in "ABC"}

for res in results:
    for k, v in res["prob_full_on_arrival"].items():
        pfull[k].append(v)

    for k, v in res["relocations"].items():
        rel[k].append(v)

    for k, v in res["mean_utilization"].items():
        util[k].append(v)

In [ ]:
def ci(x):
    x = np.array(x)
    m = x.mean()
    h = norm.ppf(.975) * x.std(ddof=1) / np.sqrt(len(x))
    return m, m-h, m+h

def show(title, data):
    print(f"\n--- {title} ---")
    for name, values in data.items():
        m, l, u = ci(values)
        print(f"{name}: {m:.3f} (95% CI: [{l:.3f}, {u:.3f}])")

show("Probability full on arrival", {
    "Ward A": pfull["A"],
    "Ward B": pfull["B"],
    "Ward C": pfull["C"]
})

show("Relocations", {
    "Ward A": rel["A"],
    "Ward B": rel["B"],
    "Ward C": rel["C"],
    "Total": rel["Total"]
})

show("Mean utilization", {
    "Ward A": util["A"],
    "Ward B": util["B"],
    "Ward C": util["C"]
})

# Plot mean utilization using CI function
plt.figure(figsize=(6,5))
util_means = {k: ci(v)[0] for k, v in util.items()}
util_lowers = {k: ci(v)[1] for k, v in util.items()}
util_uppers = {k: ci(v)[2] for k, v in util.items()}
lower_errors = [util_means["A"] - util_lowers["A"],
                util_means["B"] - util_lowers["B"],
                util_means["C"] - util_lowers["C"]]
upper_errors = [util_uppers["A"] - util_means["A"],
                util_uppers["B"] - util_means["B"],
                util_uppers["C"] - util_means["C"]]

plt.bar(["Ward A", "Ward B", "Ward C"],
        [util_means["A"], util_means["B"], util_means["C"]],
        color=["#4C72B0", "#55A868", "#C44E52"])
plt.errorbar(["Ward A", "Ward B", "Ward C"],
             [util_means["A"], util_means["B"], util_means["C"]],
             yerr=np.array([lower_errors, upper_errors]),
             fmt='none', ecolor='black', capsize=5)
plt.ylabel("Mean Utilization")
plt.title("Mean Utilization with 95% CI")
plt.show()

## Sensitivity analysis

In [ ]:
class Ward_exp:
    def __init__(self, name, arrival_func, u, beds=None):
        self.name = name
        self.arrival_func = arrival_func 
        self.u = u
        self.beds = beds

    def arrival_rate(self, t):
        return self.arrival_func(t)

    def length_of_stay(self):
        return np.random.exponential(self.u)

In [ ]:
np.random.seed(42)

n = 365
total_beds = 75

mean_a = 8
mean_b = 12
mean_c = 10

wards_exp = [
    Ward_exp("Ward A", arrival_func=arrival_a, u=mean_a, beds=0),
    Ward_exp("Ward B", arrival_func=arrival_b, u=mean_b, beds=0),
    Ward_exp("Ward C", arrival_func=arrival_c, u=mean_c, beds=0)
]

dist = get_all_bed_distributions(total_beds)

optimal_beds, min_relocated = find_optimal_distribution(
    dist, *wards_exp, replications=1
)

print("\n--- OPTIMAL CONFIGURATION FOUND ---")
print("\n".join(
    f"Ward {chr(65+i)} Beds: {beds}"
    for i, beds in enumerate(optimal_beds)
))
print(f"Minimum Average Relocated Patients: {min_relocated:.2f}")

for ward, beds in zip(wards_exp, optimal_beds):
    ward.beds = beds

print("\nFinal Results for Optimal Distribution:")
final = simulate_system(*wards_exp, n_days=n)
print(final)

In [ ]:
def extract_metrics(result):
    return {
        "reloc_A": result["relocations"]["A"],
        "reloc_B": result["relocations"]["B"],
        "reloc_C": result["relocations"]["C"],
        "reloc_total": result["relocations"]["Total"],
        "full_A": result["prob_full_on_arrival"]["A"],
        "full_B": result["prob_full_on_arrival"]["B"],
        "full_C": result["prob_full_on_arrival"]["C"],
        "util_A": result["mean_utilization"]["A"],
        "util_B": result["mean_utilization"]["B"],
        "util_C": result["mean_utilization"]["C"],
    }

lognormal_metrics = []
exp_metrics = []

np.random.seed(42)
for beds in dist:
    for ward, beds_amount in zip(wards, beds):
        ward.beds = beds_amount
    res = simulate_system(*wards, n_days=365)
    lognormal_metrics.append(extract_metrics(res))

np.random.seed(42)
for beds in dist:
    for ward, beds_amount in zip(wards_exp, beds):
        ward.beds = beds_amount
    res = simulate_system(*wards_exp, n_days=365)
    exp_metrics.append(extract_metrics(res))

import numpy as np

lognormal_arr = {k: np.array([m[k] for m in lognormal_metrics]) for k in lognormal_metrics[0]}
exp_arr       = {k: np.array([m[k] for m in exp_metrics])       for k in exp_metrics[0]}

In [ ]:
from scipy.stats import ks_2samp

for key in lognormal_arr:
    ks_stat, p = ks_2samp(lognormal_arr[key], exp_arr[key])
    print(f"{key}: KS={ks_stat:.4f}, p={p:.4g}")

from scipy.stats import chisquare
import numpy as np

def chi_square_test(x, y, bins=20):
    # fælles bins
    bins = np.linspace(min(x.min(), y.min()),
                       max(x.max(), y.max()),
                       bins)

    hist_x, _ = np.histogram(x, bins=bins)
    hist_y, _ = np.histogram(y, bins=bins)

    # chi-square test
    chi2, p = chisquare(f_obs=hist_x, f_exp=hist_y)
    return chi2, p

for key in lognormal_arr:
    chi2, p = chi_square_test(lognormal_arr[key], exp_arr[key])
    print(f"{key}: chi2={chi2:.3f}, p={p:.4g}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

n = 365
total_beds_list = [25, 35, 45, 55, 65, 75, 85, 95, 105, 115, 125]

# Arrays to store results
optimal_As, optimal_Bs, optimal_Cs = [], [], []
reloc_totals = []
probA_list, probB_list, probC_list = [], [], []
utilA_list, utilB_list, utilC_list = [], [], []

for total_beds in total_beds_list:
    print(f"\n--- Testing Total Beds: {total_beds} ---")

    # Reset wards for each run
    wards = [
        Ward("Ward A", arrival_func=arrival_a, u=mu_a, std=std, beds=0),
        Ward("Ward B", arrival_func=arrival_b, u=mu_b, std=std, beds=0),
        Ward("Ward C", arrival_func=arrival_c, u=mu_c, std=std, beds=0)
    ]

    dist = get_all_bed_distributions(total_beds)
    optimal_beds, min_relocated = find_optimal_distribution(
        dist, *wards, replications=1
    )

    # Save optimal bed allocation
    a, b, c = optimal_beds
    optimal_As.append(a)
    optimal_Bs.append(b)
    optimal_Cs.append(c)

    # Run final simulation with optimal config
    for ward, beds in zip(wards, optimal_beds):
        ward.beds = beds

    final = simulate_system(*wards, n_days=n)

    # Save performance metrics
    reloc_totals.append(final["relocations"]["Total"])
    probA_list.append(final["prob_full_on_arrival"]["A"])
    probB_list.append(final["prob_full_on_arrival"]["B"])
    probC_list.append(final["prob_full_on_arrival"]["C"])
    utilA_list.append(final["mean_utilization"]["A"])
    utilB_list.append(final["mean_utilization"]["B"])
    utilC_list.append(final["mean_utilization"]["C"])


In [ ]:
print("Total Beds:", total_beds_list)
print(reloc_totals)
print(probA_list)
print(probB_list)
print(probC_list)
print(utilA_list)
print(utilB_list)
print(utilC_list)
print(optimal_As)
print(optimal_Bs)
print(optimal_Cs)

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(total_beds_list, optimal_As, marker='o', label="Ward A")
plt.plot(total_beds_list, optimal_Bs, marker='o', label="Ward B")
plt.plot(total_beds_list, optimal_Cs, marker='o', label="Ward C")

plt.xlabel("Total beds")
plt.ylabel("Optimal bed allocation")
plt.title("Optimal bed allocation as a function of total beds")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(8,5))
plt.plot(total_beds_list, reloc_totals, marker='o', color='black')

plt.xlabel("Total beds")
plt.ylabel("Total relocations")
plt.title("Total relocations as a function of total beds")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(8,5))
plt.plot(total_beds_list, probA_list, marker='o', label="Ward A")
plt.plot(total_beds_list, probB_list, marker='o', label="Ward B")
plt.plot(total_beds_list, probC_list, marker='o', label="Ward C")

plt.xlabel("Total beds")
plt.ylabel("Probability full on arrival")
plt.title("Blocking probability vs total beds")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(8,5))
plt.plot(total_beds_list, utilA_list, marker='o', label="Ward A")
plt.plot(total_beds_list, utilB_list, marker='o', label="Ward B")
plt.plot(total_beds_list, utilC_list, marker='o', label="Ward C")

plt.xlabel("Total beds")
plt.ylabel("Mean utilization")
plt.title("Utilization vs total beds")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()
